In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from typing import List, Dict, Tuple, Optional
import json
import networkx as nx
from collections import defaultdict

c:\Users\abhin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# For GNNExplainer
try:
    from torch_geometric.explain import Explainer, GNNExplainer as TG_GNNExplainer
    USE_NEW_EXPLAINER = True
except ImportError:
    from torch_geometric.nn import GNNExplainer as TG_GNNExplainer
    USE_NEW_EXPLAINER = False

# For SHAP (optional, install with: pip install shap)
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️  SHAP not installed. Install with: pip install shap")

In [ ]:
class GNNWithAttention(torch.nn.Module):
    """Modified GNN that returns attention weights for XAI"""
    def __init__(self, in_channels):
        super(GNNWithAttention, self).__init__()
        self.conv1 = GATConv(in_channels, 64, heads=4)
        self.conv2 = GATConv(64*4, in_channels, heads=1)

    def forward(self, x, edge_index, return_attention_weights=False):
        if return_attention_weights:
            x, (edge_idx1, attn1) = self.conv1(x, edge_index, return_attention_weights=True)
            x = F.relu(x)
            x, (edge_idx2, attn2) = self.conv2(x, edge_index, return_attention_weights=True)
            return x, [(edge_idx1, attn1), (edge_idx2, attn2)]
        else:
            x = F.relu(self.conv1(x, edge_index))
            x = self.conv2(x, edge_index)
            return x

In [15]:
class GNNXAIAnalyzer:
    """Complete XAI analysis suite for GNN Recommendation System"""
    
    def __init__(self, csv_path: str, model_path: str):
        self.csv_path = csv_path
        self.model_path = model_path
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"🖥️  Using device: {self.device}")
        
        # Load data
        self.load_and_prepare_data()
        
        # Load trained model
        self.load_trained_model()
        
    def load_and_prepare_data(self):
        """Load and prepare data (same as original system)"""
        print("📊 Loading data...")
        self.df = pd.read_csv(self.csv_path)
        self.model_embed = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Map categorical features to scores
        def map_performance(perf):
            if "Above 80%" in str(perf):
                return 4
            elif "Between 65-80%" in str(perf):
                return 3
            elif "Between 50-65%" in str(perf):
                return 2
            elif "Less than 50%" in str(perf):
                return 1
            return 0
        
        def map_stress(stress):
            stress_str = str(stress).lower()
            if "frequently" in stress_str or "daily" in stress_str:
                return 3
            elif "occasionally" in stress_str or "weekly" in stress_str:
                return 2
            elif "rarely" in stress_str:
                return 1
            return 0
            
        def map_confidence(conf):
            conf_str = str(conf).lower()
            if "very confident" in conf_str:
                return 3
            elif "confident" in conf_str:
                return 2
            elif "slightly confident" in conf_str:
                return 1
            return 0
        
        self.df["performance_score"] = self.df["What was your performance in previous academic levels?"].apply(map_performance)
        self.df["stress_score"] = self.df["How often do you feel stressed about academics?"].apply(map_stress)
        self.df["confidence_score"] = self.df["How confident are you in your academic abilities?"].apply(map_confidence)
        
        # Create embeddings
        self.combined_answers = self.df.apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
        with torch.no_grad():
            embeddings = self.model_embed.encode(self.combined_answers.tolist(), convert_to_tensor=True)
        embeddings = embeddings.cpu()
        
        # Build graph
        similarity_matrix = cosine_similarity(embeddings.numpy())
        threshold = 0.6
        edge_index = []
        for i in range(len(similarity_matrix)):
            for j in range(len(similarity_matrix)):
                if similarity_matrix[i][j] > threshold and i != j:
                    edge_index.append([i, j])
        
        # Prepare features
        perf = torch.tensor(self.df["performance_score"].values, dtype=torch.float).unsqueeze(1)
        stress = torch.tensor(self.df["stress_score"].values, dtype=torch.float).unsqueeze(1)
        conf = torch.tensor(self.df["confidence_score"].values, dtype=torch.float).unsqueeze(1)
        x = torch.cat([embeddings, perf, stress, conf], dim=1)
        
        self.edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous().to(self.device)
        self.features = x.to(self.device)
        self.data = Data(x=self.features, edge_index=self.edge_index)
        
        print(f"✅ Data loaded: {len(self.df)} students, {self.edge_index.shape[1]} edges")
        
    def load_trained_model(self):
        """Load the trained model from saved weights"""
        print(f"🔄 Loading trained model from {self.model_path}...")
        
        # Load saved weights first to check version
        state_dict = torch.load(self.model_path, map_location=self.device)
        
        # Check if model uses old PyG format (lin_src/lin_dst) or new format (lin)
        uses_old_format = any('lin_src' in key or 'lin_dst' in key for key in state_dict.keys())
        
        if uses_old_format:
            print("   Detected older PyG GATConv format (lin_src/lin_dst)")
            print("   ⚠️  Your model was trained with an older PyTorch Geometric version")
            print("   💡 Recommendation: Use the same environment where you trained the model")
            print("   🔄 Attempting to load anyway with parameter matching...")
            
            # Just use the original GNN structure without trying to convert
            # Load with strict=False and let some parameters be missing
            class GNN(torch.nn.Module):
                def __init__(self, in_channels):
                    super(GNN, self).__init__()
                    self.conv1 = GATConv(in_channels, 64, heads=4)
                    self.conv2 = GATConv(64*4, in_channels, heads=1)

                def forward(self, x, edge_index, return_attention_weights=False):
                    if return_attention_weights:
                        x, (edge_idx1, attn1) = self.conv1(x, edge_index, return_attention_weights=True)
                        x = F.relu(x)
                        x, (edge_idx2, attn2) = self.conv2(x, edge_index, return_attention_weights=True)
                        return x, [(edge_idx1, attn1), (edge_idx2, attn2)]
                    else:
                        x = F.relu(self.conv1(x, edge_index))
                        x = self.conv2(x, edge_index)
                        return x
            
            self.model = GNN(in_channels=self.features.shape[1]).to(self.device)
            
            # Try to initialize weights randomly first
            print("   🎲 Initializing model with random weights first...")
            
            # Then try to load whatever matches
            result = self.model.load_state_dict(state_dict, strict=False)
            
            print(f"   ⚠️  Missing keys: {len(result.missing_keys)}")
            print(f"   ⚠️  Unexpected keys: {len(result.unexpected_keys)}")
            
            if len(result.missing_keys) > 0:
                print("\n   ❌ INCOMPATIBLE MODEL FORMAT")
                print("   The saved model uses a different PyTorch Geometric version than your current environment.")
                print("\n   🔧 SOLUTIONS:")
                print("   1. Retrain the model in your current environment")
                print("   2. OR install the exact PyTorch Geometric version used during training")
                print("   3. OR run this code in the same environment where you trained the model")
                raise RuntimeError("Model architecture mismatch - cannot load weights")
        else:
            print("   Detected newer PyG GATConv format (lin)")
            
            class GNN(torch.nn.Module):
                def __init__(self, in_channels):
                    super(GNN, self).__init__()
                    self.conv1 = GATConv(in_channels, 64, heads=4)
                    self.conv2 = GATConv(64*4, in_channels, heads=1)

                def forward(self, x, edge_index, return_attention_weights=False):
                    if return_attention_weights:
                        x, (edge_idx1, attn1) = self.conv1(x, edge_index, return_attention_weights=True)
                        x = F.relu(x)
                        x, (edge_idx2, attn2) = self.conv2(x, edge_index, return_attention_weights=True)
                        return x, [(edge_idx1, attn1), (edge_idx2, attn2)]
                    else:
                        x = F.relu(self.conv1(x, edge_index))
                        x = self.conv2(x, edge_index)
                        return x
            
            self.model = GNN(in_channels=self.features.shape[1]).to(self.device)
            self.model.load_state_dict(state_dict)
        
        self.model.eval()  # Set to evaluation mode
        
        print("✅ Model loaded successfully!")
        
    # ==================== XAI Method 1: Attention Weight Visualization ====================
    
    def extract_attention_weights(self):
        """Extract attention weights from GAT layers"""
        print("\n" + "="*70)
        print("🔍 XAI Method 1: Attention Weight Analysis")
        print("="*70)
        
        self.model.eval()
        with torch.no_grad():
            _, attention_weights = self.model(self.data.x, self.data.edge_index, return_attention_weights=True)
        
        # attention_weights is a list of tuples: [(edge_index, attention), ...]
        layer1_edges, layer1_attn = attention_weights[0]
        layer2_edges, layer2_attn = attention_weights[1]
        
        print(f"📊 Layer 1 - Attention shape: {layer1_attn.shape}")
        print(f"   Mean attention: {layer1_attn.mean().item():.4f}, Std: {layer1_attn.std().item():.4f}")
        print(f"📊 Layer 2 - Attention shape: {layer2_attn.shape}")
        print(f"   Mean attention: {layer2_attn.mean().item():.4f}, Std: {layer2_attn.std().item():.4f}")
        
        return {
            'layer1': {'edges': layer1_edges, 'attention': layer1_attn},
            'layer2': {'edges': layer2_edges, 'attention': layer2_attn}
        }
    
    def visualize_attention_for_node(self, node_idx: int, attention_data: dict, top_k: int = 10):
        """Visualize top-k attention weights for a specific node"""
        print(f"\n📍 Analyzing attention for Student {node_idx}")
        
        # Get layer 1 attention (more interpretable)
        edges = attention_data['layer1']['edges'].cpu()
        attn = attention_data['layer1']['attention'].cpu()
        
        # Find edges where node_idx is the target
        mask = edges[1] == node_idx
        source_nodes = edges[0][mask].numpy()
        attention_scores = attn[mask].mean(dim=1).numpy()  # Average over heads
        
        if len(source_nodes) == 0:
            print(f"   No incoming edges for student {node_idx}")
            return
        
        # Get top-k
        top_indices = np.argsort(attention_scores)[-top_k:][::-1]
        top_sources = source_nodes[top_indices]
        top_scores = attention_scores[top_indices]
        
        print(f"\n   Top {len(top_sources)} students influencing Student {node_idx}:")
        print(f"   {'Student':<10} {'Attention':<12} {'Performance':<12} {'Stress':<10}")
        print("   " + "-"*50)
        
        for src, score in zip(top_sources, top_scores):
            perf = self.df.iloc[src]['performance_score']
            stress = self.df.iloc[src]['stress_score']
            print(f"   {src:<10} {score:<12.4f} {perf:<12} {stress:<10}")
        
        return {'sources': top_sources, 'scores': top_scores}
    
    def plot_attention_distribution(self, attention_data: dict, save_path: str = 'attention_distribution.png'):
        """Plot distribution of attention weights"""
        layer1_attn = attention_data['layer1']['attention'].cpu().mean(dim=1).numpy()
        layer2_attn = attention_data['layer2']['attention'].cpu().mean(dim=1).numpy()
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        axes[0].hist(layer1_attn, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
        axes[0].set_xlabel('Attention Weight')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Layer 1 Attention Distribution')
        axes[0].grid(True, alpha=0.3)
        
        axes[1].hist(layer2_attn, bins=50, alpha=0.7, color='salmon', edgecolor='black')
        axes[1].set_xlabel('Attention Weight')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Layer 2 Attention Distribution')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"💾 Saved attention distribution plot to {save_path}")
        plt.close()
    
    # ==================== XAI Method 2: GNNExplainer ====================
    
    def apply_gnn_explainer(self, node_idx: int):
        """Apply GNNExplainer to explain predictions for a specific node"""
        print("\n" + "="*70)
        print("🔍 XAI Method 2: GNNExplainer")
        print("="*70)
        print(f"📍 Explaining recommendation for Student {node_idx}")
        
        try:
            if USE_NEW_EXPLAINER:
                # PyG 2.3+ API
                explainer = Explainer(
                    model=self.model,
                    algorithm=TG_GNNExplainer(epochs=200),
                    explanation_type='model',
                    node_mask_type='attributes',
                    edge_mask_type='object',
                    model_config=dict(
                        mode='regression',
                        task_level='node',
                        return_type='raw',
                    ),
                )
                explanation = explainer(self.data.x, self.data.edge_index, index=node_idx)
                edge_mask = explanation.edge_mask
                node_feat_mask = explanation.node_mask
            else:
                # Older PyG API
                explainer = TG_GNNExplainer(self.model, epochs=200, return_type='raw')
                node_feat_mask, edge_mask = explainer.explain_node(node_idx, self.data.x, self.data.edge_index)
            
            # Analyze edge importance
            edges = self.data.edge_index.cpu().numpy()
            edge_mask_np = edge_mask.cpu().detach().numpy()
            
            # Find edges connected to target node
            mask = (edges[1] == node_idx) | (edges[0] == node_idx)
            relevant_edges = edges[:, mask]
            relevant_importance = edge_mask_np[mask]
            
            # Get top important edges
            top_k = min(10, len(relevant_importance))
            top_indices = np.argsort(relevant_importance)[-top_k:][::-1]
            
            print(f"\n   Top {top_k} most important connections:")
            print(f"   {'From':<8} {'To':<8} {'Importance':<12} {'Performance Diff':<15}")
            print("   " + "-"*50)
            
            for idx in top_indices:
                src, tgt = relevant_edges[0, idx], relevant_edges[1, idx]
                importance = relevant_importance[idx]
                perf_diff = abs(self.df.iloc[src]['performance_score'] - 
                              self.df.iloc[tgt]['performance_score'])
                print(f"   {src:<8} {tgt:<8} {importance:<12.4f} {perf_diff:<15.1f}")
            
            # Analyze feature importance
            print(f"\n   Feature importance (last 3 dimensions are performance, stress, confidence):")
            feat_importance = node_feat_mask.cpu().detach().numpy()
            
            # Aggregate embeddings vs structured features
            embed_importance = feat_importance[:-3].mean()
            perf_importance = feat_importance[-3]
            stress_importance = feat_importance[-2]
            conf_importance = feat_importance[-1]
            
            print(f"   Embedding features: {embed_importance:.4f}")
            print(f"   Performance score:  {perf_importance:.4f}")
            print(f"   Stress score:       {stress_importance:.4f}")
            print(f"   Confidence score:   {conf_importance:.4f}")
            
            return {
                'edge_mask': edge_mask,
                'node_feat_mask': node_feat_mask,
                'top_edges': (relevant_edges[:, top_indices], relevant_importance[top_indices])
            }
            
        except Exception as e:
            print(f"❌ GNNExplainer failed: {e}")
            print("   This might be due to PyTorch Geometric version compatibility")
            return None
    
    # ==================== XAI Method 3: Feature Attribution (Saliency Maps) ====================
    
    def compute_feature_attribution(self, node_idx: int):
        """Compute feature attribution using gradient-based saliency"""
        print("\n" + "="*70)
        print("🔍 XAI Method 3: Feature Attribution (Saliency Maps)")
        print("="*70)
        print(f"📍 Computing feature attribution for Student {node_idx}")
        
        self.model.eval()
        
        # Make input require gradients
        x = self.data.x.clone().detach().requires_grad_(True)
        
        # Forward pass
        output = self.model(x, self.data.edge_index)
        
        # Get output for target node
        node_output = output[node_idx]
        
        # Compute gradients
        node_output.sum().backward()
        
        # Get gradients (saliency)
        saliency = x.grad[node_idx].abs().cpu().numpy()
        
        # Analyze feature importance
        embed_saliency = saliency[:-3].mean()
        perf_saliency = saliency[-3]
        stress_saliency = saliency[-2]
        conf_saliency = saliency[-1]
        
        print(f"\n   Feature Saliency (importance):")
        print(f"   Embedding features:  {embed_saliency:.4f}")
        print(f"   Performance score:   {perf_saliency:.4f}")
        print(f"   Stress score:        {stress_saliency:.4f}")
        print(f"   Confidence score:    {conf_saliency:.4f}")
        
        # Find most important embedding dimensions
        top_k = 5
        embed_dims = saliency[:-3]
        top_dims = np.argsort(embed_dims)[-top_k:][::-1]
        
        print(f"\n   Top {top_k} most important embedding dimensions:")
        for i, dim in enumerate(top_dims, 1):
            print(f"   {i}. Dimension {dim}: {embed_dims[dim]:.4f}")
        
        return {
            'saliency': saliency,
            'structured_features': {
                'performance': perf_saliency,
                'stress': stress_saliency,
                'confidence': conf_saliency,
                'embeddings': embed_saliency
            }
        }
    
    def plot_feature_attribution(self, attribution_data: dict, save_path: str = 'feature_attribution.png'):
        """Plot feature attribution results"""
        struct_feats = attribution_data['structured_features']
        
        labels = ['Embeddings', 'Performance', 'Stress', 'Confidence']
        values = [
            struct_feats['embeddings'],
            struct_feats['performance'],
            struct_feats['stress'],
            struct_feats['confidence']
        ]
        
        plt.figure(figsize=(10, 6))
        bars = plt.bar(labels, values, color=['#3498db', '#e74c3c', '#f39c12', '#2ecc71'])
        plt.xlabel('Feature Type', fontsize=12)
        plt.ylabel('Attribution Score', fontsize=12)
        plt.title('Feature Attribution Analysis', fontsize=14, fontweight='bold')
        plt.grid(axis='y', alpha=0.3)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.4f}',
                    ha='center', va='bottom', fontsize=10)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"💾 Saved feature attribution plot to {save_path}")
        plt.close()
    
    # ==================== XAI Method 4: Integrated Gradients ====================
    
    def compute_integrated_gradients(self, node_idx: int, steps: int = 50):
        """Compute Integrated Gradients for feature attribution"""
        print("\n" + "="*70)
        print("🔍 XAI Method 4: Integrated Gradients")
        print("="*70)
        print(f"📍 Computing integrated gradients for Student {node_idx} with {steps} steps")
        
        self.model.eval()
        
        # Original input
        x_original = self.data.x[node_idx].clone().detach()
        
        # Baseline (zeros)
        x_baseline = torch.zeros_like(x_original)
        
        # Interpolate between baseline and input
        integrated_grads = torch.zeros_like(x_original)
        
        for i in range(steps):
            # Interpolate
            alpha = i / steps
            x_interpolated = x_baseline + alpha * (x_original - x_baseline)
            
            # Replace node features with interpolated
            x_batch = self.data.x.clone()
            x_batch[node_idx] = x_interpolated
            x_batch.requires_grad_(True)
            
            # Forward pass
            output = self.model(x_batch, self.data.edge_index)
            node_output = output[node_idx].sum()
            
            # Backward pass
            node_output.backward()
            
            # Accumulate gradients
            integrated_grads += x_batch.grad[node_idx]
        
        # Average and multiply by input difference
        integrated_grads = integrated_grads / steps
        integrated_grads = integrated_grads * (x_original - x_baseline)
        integrated_grads = integrated_grads.abs().cpu().numpy()
        
        # Analyze results
        embed_ig = integrated_grads[:-3].mean()
        perf_ig = integrated_grads[-3]
        stress_ig = integrated_grads[-2]
        conf_ig = integrated_grads[-1]
        
        print(f"\n   Integrated Gradients Attribution:")
        print(f"   Embedding features:  {embed_ig:.4f}")
        print(f"   Performance score:   {perf_ig:.4f}")
        print(f"   Stress score:        {stress_ig:.4f}")
        print(f"   Confidence score:    {conf_ig:.4f}")
        
        return {
            'integrated_gradients': integrated_grads,
            'structured_features': {
                'performance': perf_ig,
                'stress': stress_ig,
                'confidence': conf_ig,
                'embeddings': embed_ig
            }
        }
    
    # ==================== XAI Method 5: SHAP Values ====================
    
    def compute_shap_values(self, node_idx: int, background_samples: int = 10):
        """Compute SHAP values for structured features"""
        if not SHAP_AVAILABLE:
            print("\n⚠️  SHAP not available. Install with: pip install shap")
            return None
        
        print("\n" + "="*70)
        print("🔍 XAI Method 5: SHAP Values")
        print("="*70)
        print(f"📍 Computing SHAP values for Student {node_idx}")
        
        # Extract structured features only (performance, stress, confidence)
        structured_features = self.data.x[:, -3:].cpu().numpy()
        target_features = structured_features[node_idx:node_idx+1]
        
        # Define prediction function for SHAP
        def predict_fn(features):
            # Create modified feature tensor
            batch_size = features.shape[0]
            predictions = []
            
            for i in range(batch_size):
                x_modified = self.data.x.clone()
                x_modified[node_idx, -3:] = torch.tensor(features[i], dtype=torch.float).to(self.device)
                
                with torch.no_grad():
                    output = self.model(x_modified, self.data.edge_index)
                predictions.append(output[node_idx].cpu().numpy())
            
            return np.array(predictions)
        
        # Sample background data
        background_idx = np.random.choice(len(structured_features), 
                                         size=min(background_samples, len(structured_features)), 
                                         replace=False)
        background = structured_features[background_idx]
        
        # Compute SHAP values
        explainer = shap.KernelExplainer(predict_fn, background)
        shap_values = explainer.shap_values(target_features)
        
        # Handle different SHAP output formats
        if isinstance(shap_values, list):
            shap_vals = shap_values[0] if len(shap_values) > 0 else shap_values
        else:
            shap_vals = shap_values
        
        # Ensure it's a 1D array
        if len(shap_vals.shape) > 1:
            shap_vals = shap_vals.flatten()
        
        print(f"\n   SHAP Values:")
        print(f"   Performance score:  {float(shap_vals[0]):.4f}")
        print(f"   Stress score:       {float(shap_vals[1]):.4f}")
        print(f"   Confidence score:   {float(shap_vals[2]):.4f}")
        
        return {
            'shap_values': shap_vals,
            'feature_values': target_features[0],
            'feature_names': ['Performance', 'Stress', 'Confidence']
        }
    
    # ==================== XAI Method 6: Qualitative Feature Analysis ====================
    
    def analyze_qualitative_features(self, node_idx: int, top_similar_nodes: List[int] = None):
        """Analyze qualitative/categorical features that influenced recommendations"""
        print("\n" + "="*70)
        print("🔍 XAI Method 6: Qualitative Feature Analysis")
        print("="*70)
        print(f"📍 Analyzing qualitative patterns for Student {node_idx}\n")
        
        # Key qualitative columns to analyze
        qualitative_cols = [
            'What is your preferred mode of learning?',
            'How do you usually take notes?',
            'How do you retain information best?',
            'What type of academic challenges do you face the most?',
            'How do you handle academic stress?',
            'What study technique do you use the most?',
            'What keeps you motivated to study consistently?',
            'What learning activities do you find most engaging?'
        ]
        
        target_student = self.df.iloc[node_idx]
        
        print("   📋 Target Student Profile:")
        print("   " + "-"*65)
        for col in qualitative_cols:
            if col in self.df.columns:
                value = str(target_student[col])[:80]  # Truncate long responses
                print(f"   {col[:40]:<40}: {value}")
        
        # If we have similar nodes from attention analysis, compare with them
        if top_similar_nodes is not None and len(top_similar_nodes) > 0:
            print("\n   🔄 Comparing with Top Similar Students:")
            print("   " + "-"*65)
            
            # Analyze patterns in similar students
            feature_patterns = {}
            for col in qualitative_cols:
                if col not in self.df.columns:
                    continue
                    
                # Get target's value
                target_val = str(target_student[col])
                
                # Get similar students' values
                similar_vals = [str(self.df.iloc[idx][col]) for idx in top_similar_nodes if idx < len(self.df)]
                
                # Count matches
                matches = sum(1 for v in similar_vals if target_val.lower() in v.lower() or v.lower() in target_val.lower())
                match_rate = (matches / len(similar_vals)) * 100 if similar_vals else 0
                
                feature_patterns[col] = {
                    'target': target_val[:60],
                    'match_rate': match_rate,
                    'common_responses': self._get_common_patterns(similar_vals)
                }
            
            # Show most aligned features
            sorted_features = sorted(feature_patterns.items(), key=lambda x: x[1]['match_rate'], reverse=True)
            
            print(f"\n   🎯 Most Aligned Features (with similar students):")
            for col, data in sorted_features[:5]:
                if data['match_rate'] > 0:
                    print(f"\n   • {col[:50]}")
                    print(f"     Match Rate: {data['match_rate']:.1f}%")
                    print(f"     Target: {data['target']}")
                    if data['common_responses']:
                        print(f"     Common in group: {data['common_responses'][:60]}")
            
            print(f"\n   🔀 Most Diverse Features (different from similar students):")
            for col, data in sorted_features[-3:]:
                print(f"\n   • {col[:50]}")
                print(f"     Match Rate: {data['match_rate']:.1f}%")
                print(f"     Target: {data['target']}")
        
        return feature_patterns if top_similar_nodes else None
    
    def _get_common_patterns(self, values: List[str]) -> str:
        """Extract common patterns from list of string values"""
        if not values:
            return ""
        
        # For multi-choice responses separated by ' / '
        all_choices = []
        for v in values:
            if ' / ' in v:
                all_choices.extend([c.strip() for c in v.split(' / ')])
            else:
                all_choices.append(v.strip())
        
        # Count frequency
        from collections import Counter
        counts = Counter(all_choices)
        if counts:
            most_common = counts.most_common(1)[0][0]
            return most_common[:60]
        return ""
    
    # ==================== XAI Method 7: Text Embedding Interpretation ====================
    
    def analyze_embedding_importance(self, node_idx: int, top_k: int = 10):
        """Analyze which parts of the text embeddings are most important"""
        print("\n" + "="*70)
        print("🔍 XAI Method 7: Text Embedding Interpretation")
        print("="*70)
        print(f"📍 Analyzing semantic patterns for Student {node_idx}\n")
        
        # Get the student's combined text
        student_text = self.combined_answers.iloc[node_idx]
        
        # Get feature attribution for embeddings
        self.model.eval()
        x = self.data.x.clone().detach().requires_grad_(True)
        output = self.model(x, self.data.edge_index)
        node_output = output[node_idx]
        node_output.sum().backward()
        
        # Get embedding gradients (first 384 dimensions for MiniLM)
        embedding_grads = x.grad[node_idx][:-3].abs().cpu().numpy()
        
        # Find most important embedding dimensions
        top_dims = np.argsort(embedding_grads)[-top_k:][::-1]
        
        print(f"   📊 Top {top_k} Most Important Embedding Dimensions:")
        print(f"   {'Dimension':<12} {'Importance':<12} {'Context'}")
        print("   " + "-"*65)
        
        for i, dim in enumerate(top_dims[:top_k], 1):
            importance = embedding_grads[dim]
            print(f"   Dim {dim:<7} {importance:<12.6f}")
        
        # Analyze word-level importance (approximate)
        print(f"\n   📝 Analyzing Text Content:")
        words = student_text.split()[:100]  # First 100 words
        print(f"   Total words in profile: {len(student_text.split())}")
        print(f"   Embedding vector length: {len(embedding_grads)}")
        print(f"   Mean embedding importance: {embedding_grads.mean():.6f}")
        print(f"   Std embedding importance: {embedding_grads.std():.6f}")
        
        # Find key phrases (very rough approximation)
        print(f"\n   🔑 Sample of student's profile text:")
        sample_text = ' '.join(words[:50])
        print(f"   \"{sample_text}...\"")
        
        return {
            'top_dimensions': top_dims.tolist(),
            'importance_scores': embedding_grads[top_dims].tolist(),
            'embedding_stats': {
                'mean': float(embedding_grads.mean()),
                'std': float(embedding_grads.std()),
                'max': float(embedding_grads.max())
            }
        }
    
    # ==================== XAI Method 8: Learning Pattern Clustering ====================
    
    def analyze_learning_patterns(self, node_idx: int):
        """Cluster students by learning patterns and show where target student fits"""
        print("\n" + "="*70)
        print("🔍 XAI Method 8: Learning Pattern Clustering")
        print("="*70)
        print(f"📍 Finding learning pattern clusters for Student {node_idx}\n")
        
        target_student = self.df.iloc[node_idx]
        
        # Define learning pattern categories
        patterns = {
            'Learning Style': 'What is your preferred mode of learning?',
            'Note-Taking': 'How do you usually take notes?',
            'Retention Method': 'How do you retain information best?',
            'Problem-Solving': 'How do you approach problem-solving in academics?',
            'Exam Prep': 'What is your preferred way of preparing for exams?',
            'Study Environment': 'What type of environment helps you focus while studying?',
            'Motivation': 'What keeps you motivated to study consistently?'
        }
        
        print("   👤 Target Student's Learning Profile:")
        print("   " + "-"*65)
        
        target_profile = {}
        for category, col in patterns.items():
            if col in self.df.columns:
                value = str(target_student[col])
                target_profile[category] = value
                # Extract first preference if multiple choices
                first_pref = value.split(' / ')[0] if ' / ' in value else value
                print(f"   {category:<20}: {first_pref[:60]}")
        
        # Find students with similar patterns
        print(f"\n   🔍 Finding students with similar learning patterns...")
        
        similarity_scores = []
        for idx in range(len(self.df)):
            if idx == node_idx:
                continue
            
            match_count = 0
            for category, col in patterns.items():
                if col not in self.df.columns:
                    continue
                target_val = str(target_student[col]).lower()
                student_val = str(self.df.iloc[idx][col]).lower()
                
                # Check for overlap in multi-choice responses
                target_choices = set(target_val.split(' / '))
                student_choices = set(student_val.split(' / '))
                
                if target_choices & student_choices:  # Intersection
                    match_count += 1
            
            if match_count > 0:
                similarity_scores.append((idx, match_count))
        
        # Sort by similarity
        similarity_scores.sort(key=lambda x: x[1], reverse=True)
        top_similar = similarity_scores[:5]
        
        print(f"\n   🎯 Top 5 Students with Similar Learning Patterns:")
        print(f"   {'Student':<10} {'Matches':<10} {'Performance':<12} {'Stress':<10}")
        print("   " + "-"*50)
        
        for idx, match_count in top_similar:
            perf = self.df.iloc[idx]['performance_score']
            stress = self.df.iloc[idx]['stress_score']
            print(f"   {idx:<10} {match_count}/{len(patterns):<10} {perf:<12} {stress:<10}")
        
        # Analyze successful patterns
        print(f"\n   ⭐ Learning Patterns of High Performers (performance >= 3):")
        high_performers = self.df[self.df['performance_score'] >= 3]
        
        for category, col in list(patterns.items())[:3]:  # Top 3 categories
            if col not in self.df.columns:
                continue
            print(f"\n   {category}:")
            # Get most common responses among high performers
            responses = high_performers[col].astype(str).tolist()
            common = self._get_most_common_responses(responses, top_n=2)
            for resp, count in common:
                print(f"     • {resp[:70]} ({count} students)")
        
        return {
            'similar_students': [(idx, matches) for idx, matches in top_similar],
            'target_profile': target_profile
        }
    
    def _get_most_common_responses(self, responses: List[str], top_n: int = 3):
        """Get most common responses from a list"""
        from collections import Counter
        
        # Handle multi-choice responses
        all_choices = []
        for resp in responses:
            if ' / ' in resp:
                all_choices.extend([c.strip() for c in resp.split(' / ')])
            else:
                all_choices.append(resp.strip())
        
        # Remove 'nan' and empty
        all_choices = [c for c in all_choices if c and c != 'nan']
        
        counts = Counter(all_choices)
        return counts.most_common(top_n)
    
    def run_complete_xai_analysis(self, node_idx: int, save_dir: str = 'xai_results'):
        """Run all XAI methods and generate comprehensive report"""
        os.makedirs(save_dir, exist_ok=True)
        
        print("\n" + "="*70)
        print("🚀 RUNNING COMPLETE XAI ANALYSIS")
        print("="*70)
        print(f"Target Student: {node_idx}")
        print(f"Performance: {self.df.iloc[node_idx]['performance_score']}")
        print(f"Stress: {self.df.iloc[node_idx]['stress_score']}")
        print(f"Confidence: {self.df.iloc[node_idx]['confidence_score']}")
        
        results = {}
        
        # Method 1: Attention Weights
        attention_data = self.extract_attention_weights()
        attention_analysis = self.visualize_attention_for_node(node_idx, attention_data)
        self.plot_attention_distribution(attention_data, 
                                        save_path=os.path.join(save_dir, 'attention_distribution.png'))
        results['attention'] = attention_analysis
        
        # Get top similar students for qualitative analysis
        top_similar_students = None
        if attention_analysis and 'sources' in attention_analysis:
            top_similar_students = attention_analysis['sources'].tolist()[:5]
        
        # Method 2: GNNExplainer
        gnn_explainer_results = self.apply_gnn_explainer(node_idx)
        results['gnn_explainer'] = gnn_explainer_results
        
        # Method 3: Feature Attribution
        attribution_data = self.compute_feature_attribution(node_idx)
        self.plot_feature_attribution(attribution_data, 
                                     save_path=os.path.join(save_dir, 'feature_attribution.png'))
        results['feature_attribution'] = attribution_data
        
        # Method 4: Integrated Gradients
        ig_data = self.compute_integrated_gradients(node_idx)
        results['integrated_gradients'] = ig_data
        
        # Method 5: SHAP Values
        shap_data = self.compute_shap_values(node_idx)
        results['shap'] = shap_data
        
        # Method 6: Qualitative Feature Analysis (NEW!)
        qualitative_analysis = self.analyze_qualitative_features(node_idx, top_similar_students)
        results['qualitative'] = qualitative_analysis
        
        # Method 7: Text Embedding Interpretation (NEW!)
        embedding_analysis = self.analyze_embedding_importance(node_idx)
        results['embedding'] = embedding_analysis
        
        # Method 8: Learning Pattern Clustering (NEW!)
        pattern_analysis = self.analyze_learning_patterns(node_idx)
        results['patterns'] = pattern_analysis
        
        # Generate summary comparison
        self.generate_xai_comparison(results, node_idx, save_dir)
        
        # Generate comprehensive text report
        self.generate_text_report(results, node_idx, save_dir)
        
        print("\n" + "="*70)
        print("✅ XAI ANALYSIS COMPLETE!")
        print(f"📁 Results saved to: {save_dir}/")
        print("="*70)
        
        return results
    
    def generate_xai_comparison(self, results: dict, node_idx: int, save_dir: str):
        """Generate comparison plot of all XAI methods"""
        methods = []
        perf_scores = []
        stress_scores = []
        conf_scores = []
        
        # Collect structured feature scores from each method
        if results.get('feature_attribution'):
            methods.append('Saliency')
            feat = results['feature_attribution']['structured_features']
            perf_scores.append(feat['performance'])
            stress_scores.append(feat['stress'])
            conf_scores.append(feat['confidence'])
        
        if results.get('integrated_gradients'):
            methods.append('Int. Gradients')
            feat = results['integrated_gradients']['structured_features']
            perf_scores.append(feat['performance'])
            stress_scores.append(feat['stress'])
            conf_scores.append(feat['confidence'])
        
        if results.get('shap') and results['shap'] is not None:
            try:
                methods.append('SHAP')
                shap_vals = results['shap']['shap_values']
                # Ensure we can access the values correctly
                if isinstance(shap_vals, np.ndarray):
                    if len(shap_vals.shape) > 1:
                        shap_vals = shap_vals.flatten()
                    if len(shap_vals) >= 3:
                        perf_scores.append(abs(float(shap_vals[0])))
                        stress_scores.append(abs(float(shap_vals[1])))
                        conf_scores.append(abs(float(shap_vals[2])))
                    else:
                        # Not enough values, remove SHAP from methods
                        methods.pop()
                else:
                    # Skip if format is unexpected
                    methods.pop()
            except Exception as e:
                print(f"   ⚠️  Could not process SHAP values for comparison: {e}")
                if 'SHAP' in methods:
                    methods.pop()
        
        if not methods:
            print("   ⚠️  No methods available for comparison")
            return
        
        # Create comparison plot
        x = np.arange(len(methods))
        width = 0.25
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        bars1 = ax.bar(x - width, perf_scores, width, label='Performance', color='#e74c3c')
        bars2 = ax.bar(x, stress_scores, width, label='Stress', color='#f39c12')
        bars3 = ax.bar(x + width, conf_scores, width, label='Confidence', color='#2ecc71')
        
        ax.set_xlabel('XAI Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Attribution Score', fontsize=12, fontweight='bold')
        ax.set_title(f'XAI Methods Comparison - Student {node_idx}', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(methods)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, 'xai_comparison.png'), dpi=300, bbox_inches='tight')
        print(f"💾 Saved XAI comparison plot")
        plt.close()
    
    def generate_text_report(self, results: dict, node_idx: int, save_dir: str):
        """Generate comprehensive text report of XAI analysis"""
        report_path = os.path.join(save_dir, f'xai_report_student_{node_idx}.txt')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("="*70 + "\n")
            f.write(f"XAI ANALYSIS REPORT - Student {node_idx}\n")
            f.write("="*70 + "\n\n")
            
            # Student profile
            student = self.df.iloc[node_idx]
            f.write("STUDENT PROFILE:\n")
            f.write("-"*70 + "\n")
            f.write(f"Performance Score: {student['performance_score']}\n")
            f.write(f"Stress Score: {student['stress_score']}\n")
            f.write(f"Confidence Score: {student['confidence_score']}\n\n")
            
            # Quantitative analysis summary
            if 'feature_attribution' in results:
                f.write("\nQUANTITATIVE FEATURE IMPORTANCE:\n")
                f.write("-"*70 + "\n")
                feat = results['feature_attribution']['structured_features']
                f.write(f"Performance: {feat['performance']:.4f}\n")
                f.write(f"Stress: {feat['stress']:.4f}\n")
                f.write(f"Confidence: {feat['confidence']:.4f}\n")
                f.write(f"Embeddings: {feat['embeddings']:.4f}\n\n")
            
            # Qualitative analysis
            if 'qualitative' in results and results['qualitative']:
                f.write("\nQUALITATIVE FEATURE ANALYSIS:\n")
                f.write("-"*70 + "\n")
                sorted_features = sorted(results['qualitative'].items(), 
                                       key=lambda x: x[1]['match_rate'], reverse=True)
                for col, data in sorted_features[:5]:
                    f.write(f"\n{col}:\n")
                    f.write(f"  Match Rate: {data['match_rate']:.1f}%\n")
                    f.write(f"  Student's Response: {data['target']}\n")
                    if data['common_responses']:
                        f.write(f"  Common Response: {data['common_responses']}\n")
            
            # Learning patterns
            if 'patterns' in results:
                f.write("\n\nLEARNING PATTERN ANALYSIS:\n")
                f.write("-"*70 + "\n")
                profile = results['patterns']['target_profile']
                for category, value in profile.items():
                    first_pref = value.split(' / ')[0] if ' / ' in value else value
                    f.write(f"{category}: {first_pref[:80]}\n")
            
            # Similar students
            if 'attention' in results and results['attention']:
                f.write("\n\nMOST INFLUENTIAL SIMILAR STUDENTS:\n")
                f.write("-"*70 + "\n")
                if 'sources' in results['attention']:
                    for i, student_idx in enumerate(results['attention']['sources'][:5], 1):
                        perf = self.df.iloc[student_idx]['performance_score']
                        stress = self.df.iloc[student_idx]['stress_score']
                        f.write(f"{i}. Student {student_idx}: ")
                        f.write(f"Performance={perf}, Stress={stress}\n")
        
        print(f"💾 Saved text report to {report_path}")

In [16]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = GNNXAIAnalyzer(
        csv_path='Dataset3.csv',
        model_path='best_gnn_model.pt'
    )
    
    # Choose a student to analyze (change this to any student index)
    target_student = 0
    
    # Run complete XAI analysis
    results = analyzer.run_complete_xai_analysis(
        node_idx=target_student,
        save_dir='xai_results'
    )
    
    print("\n🎉 All XAI analyses complete!")
    print("\n📊 You can now:")
    print("   1. Check the console output for detailed explanations")
    print("   2. View generated plots in xai_results/ folder")
    print("   3. Analyze different students by changing target_student variable")
    print("   4. Compare XAI methods using the comparison plot")

🖥️  Using device: cuda
📊 Loading data...
✅ Data loaded: 350 students, 111016 edges
🔄 Loading trained model from best_gnn_model.pt...
   Detected older PyG GATConv format (lin_src/lin_dst)
   ⚠️  Your model was trained with an older PyTorch Geometric version
   💡 Recommendation: Use the same environment where you trained the model
   🔄 Attempting to load anyway with parameter matching...
   🎲 Initializing model with random weights first...
   ⚠️  Missing keys: 0
   ⚠️  Unexpected keys: 0
✅ Model loaded successfully!

🚀 RUNNING COMPLETE XAI ANALYSIS
Target Student: 0
Performance: 1
Stress: 2
Confidence: 3

🔍 XAI Method 1: Attention Weight Analysis
📊 Layer 1 - Attention shape: torch.Size([111366, 4])
   Mean attention: 0.0031, Std: 0.0020
📊 Layer 2 - Attention shape: torch.Size([111366, 1])
   Mean attention: 0.0031, Std: 0.0004

📍 Analyzing attention for Student 0

   Top 10 students influencing Student 0:
   Student    Attention    Performance  Stress    
   ----------------------------

100%|██████████| 1/1 [00:00<00:00,  1.32it/s]



   SHAP Values:
   Performance score:  0.0000
   Stress score:       -0.0000
   Confidence score:   -0.0001

🔍 XAI Method 6: Qualitative Feature Analysis
📍 Analyzing qualitative patterns for Student 0

   📋 Target Student Profile:
   -----------------------------------------------------------------
   What is your preferred mode of learning?: Visual (diagrams, videos, infographics) → I learn best through images and videos
   How do you usually take notes?          : Typed notes on a device → I prefer digital notes for organization. / Handwritten
   How do you retain information best?     : Revisiting notes multiple times → Frequent review helps me. / Creating mind maps
   What type of academic challenges do you : Exam anxiety and stress → Pressure affects my performance. / Understanding compl
   How do you handle academic stress?      : Experiment with different strategies → I explore multiple ways to solve a proble
   What study technique do you use the most: Practice quizzes and sol